# DeepDetect Experiments: convnext_base


As with CIFAKE we run five experiments per backbone. These include:  spatial-only baseline, joint-only fusion, scalar fusion, gating fusion, and gating with a frozen backbone. Evaluations remain the same: accuracy, AUC-ROC, F1, delta, and gate entropy are reported for each run, and all results are logged to the shared results CSV. Given that the tuning trials consistently converged within 5-7 epochs, we train all DeepDetect experiments for 20 epochs. Val accuracy is monitored every epoch and the best checkpoint is saved, ensuring early convergence is captured regardless of when it occurs.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell
VRAM: 25.2 GB


## Best Hyperparameters
We use the hyperparameters identified by the Optuna search 

In [2]:
BEST_BACKBONE_LR       = 4.85e-05  
BEST_LR                = 2.79e-04
BEST_FREQ_AUX_WEIGHT   = 0.500
BEST_DIVERSITY_WEIGHT  = 0.058
BEST_GATE_INIT_BIAS    = 0.215



## Shared Config

In [21]:
from config import Config
from data.deepdetect import get_deepdetect_loaders
from experiments.train import train
from experiments.evaluate import full_evaluation
from experiments.baseline_spatial_only import run_spatial_only_baseline
from experiments.baseline_freq_only import run_freq_only_baseline

BACKBONE = "convnext_base"

def make_cfg(fusion_mode, frozen=False):
    cfg = Config()
    cfg.data.deepdetect_root   = "../data/raw/deep_detect/data"
    cfg.data.dataset           = "deepdetect"
    cfg.data.image_size        = 224
    cfg.data.batch_size        = 88
    cfg.data.num_workers       = 4
    cfg.backbone.name          = BACKBONE
    cfg.backbone.pretrained    = True
    cfg.backbone.img_size      = 224
    cfg.backbone.frozen        = frozen
    cfg.fusion.mode            = fusion_mode
    cfg.train.epochs           = 20
    cfg.train.backbone_lr      = BEST_BACKBONE_LR
    cfg.train.lr               = BEST_LR
    cfg.loss.freq_aux_weight   = BEST_FREQ_AUX_WEIGHT
    cfg.fusion.diversity_weight = BEST_DIVERSITY_WEIGHT
    cfg.fusion.gate_init_bias  = BEST_GATE_INIT_BIAS
    cfg.data.jpeg_aug          = True
    cfg.data.blur_aug          = True
    cfg.data.noise_aug         = True
    cfg.data.recompression_aug = True
    cfg.data.mixed_aug         = True
    cfg.data.mixed_aug_prob    = 0.3
    cfg.experiment_name = f"dd_{BACKBONE}_{fusion_mode}{'_frozen' if frozen else ''}"
    cfg.notes           = f"DeepDetect · {BACKBONE} · {fusion_mode}{'_frozen' if frozen else ''}"
    return cfg

cfg_data = make_cfg("joint_only")
train_loader, val_loader, test_loader = get_deepdetect_loaders(cfg_data)


Train: 76,848  Val: 13,561  Test: 21,776



## Experiment 0: Spatial-Only Baseline

In [4]:
cfg0 = make_cfg("joint_only")
cfg0.experiment_name = f"dd_{BACKBONE}_spatial_only"
cfg0.notes           = f"DeepDetect spatial-only baseline, {BACKBONE}"
spatial_acc = run_spatial_only_baseline(cfg0, train_loader, val_loader, test_loader)
print(f"\nSpatial-only floor: {spatial_acc:.1%}")


Device: cuda
Training spatial-only baseline (convnext_base) for 20 epochs...
Train: 76,848  Val: 13,561


Epoch   1/20 | train_loss=0.1322 | val_acc=96.7%


Epoch   2/20 | train_loss=0.0452 | val_acc=99.6%


Epoch   3/20 | train_loss=0.0266 | val_acc=98.2%


Epoch   4/20 | train_loss=0.0209 | val_acc=99.7%


...

Epoch  16/20 | train_loss=0.0010 | val_acc=99.7%


Epoch  17/20 | train_loss=0.0007 | val_acc=99.7%


Epoch  18/20 | train_loss=0.0002 | val_acc=99.8%


Epoch  19/20 | train_loss=0.0002 | val_acc=99.7%


Epoch  20/20 | train_loss=0.0001 | val_acc=99.8%

Spatial-only results (convnext_base):
  Accuracy: 96.1%
  AUC-ROC:  0.988
  F1:       0.961
Results saved → ./results/results.csv  (dd_convnext_base_spatial_only, acc=0.9611)
Results saved to ./results/results.csv

Spatial-only floor: 96.1%



## Experiment 1: Joint-Only Baseline

In [5]:
cfg1 = make_cfg("joint_only")
print(f"Running: {cfg1.experiment_name}")
train(cfg1, train_loader, val_loader, test_loader)


In [6]:
results1 = full_evaluation(
    cfg1,
    checkpoint_path=f"./checkpoints/best_{cfg1.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_convnext_base_joint_only.pt

EVALUATION — dd_convnext_base_joint_only
Backbone: convnext_base | Fusion: joint_only | Frozen: False
  Joint accuracy:     97.0%
  AUC-ROC:            0.987
  F1:                 0.970
  Spatial-only:       97.0%
  Freq-only:          58.7%
  Delta (Δ):          +0.0%  (freq branch contribution)
Results saved → ./results/results.csv  (dd_convnext_base_joint_only, acc=0.9702)



## Experiment 2: Scalar Fusion

In [7]:
cfg2 = make_cfg("scalar")
print(f"Running: {cfg2.experiment_name}")
train(cfg2, train_loader, val_loader, test_loader)


Running: dd_convnext_base_scalar
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: dd_convnext_base_scalar
Backbone: convnext_base | Fusion: scalar | Frozen: False
Epochs: 20 | LR: 0.000279 | Batch: 88



Epoch   1/20 | train_loss=0.5192 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992
  scalars — spatial=0.506 freq=0.494
  grad norms — freq=0.0060 | spatial=0.9999
  -> Saved best val_acc=99.3%


Epoch   2/20 | train_loss=0.4019 | val_acc=98.7% | val_auc=1.000 | val_f1=0.985
  scalars — spatial=0.502 freq=0.498


Epoch   3/20 | train_loss=0.3824 | val_acc=99.4% | val_auc=1.000 | val_f1=0.994
  scalars — spatial=0.500 freq=0.500
  -> Saved best val_acc=99.4%


Epoch   4/20 | train_loss=0.3706 | val_acc=99.6% | val_auc=1.000 | val_f1=0.996
  scalars — spatial=0.499 freq=0.501
  -> Saved best val_acc=99.6%


...

Epoch  17/20 | train_loss=0.3350 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997
  scalars — spatial=0.492 freq=0.508


Epoch  18/20 | train_loss=0.3351 | val_acc=99.8% | val_auc=1.000 | val_f1=0.998
  scalars — spatial=0.492 freq=0.508
  -> Saved best val_acc=99.8%


Epoch  19/20 | train_loss=0.3346 | val_acc=99.9% | val_auc=1.000 | val_f1=0.999
  scalars — spatial=0.492 freq=0.508
  -> Saved best val_acc=99.9%


Epoch  20/20 | train_loss=0.3342 | val_acc=99.8% | val_auc=1.000 | val_f1=0.998
  scalars — spatial=0.492 freq=0.508

Training complete. Best val accuracy: 99.9%
Results will be logged to CSV after full_evaluation().


0.9986726642577981

In [8]:
results2 = full_evaluation(
    cfg2,
    checkpoint_path=f"./checkpoints/best_{cfg2.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_convnext_base_scalar.pt

EVALUATION — dd_convnext_base_scalar
Backbone: convnext_base | Fusion: scalar | Frozen: False
  Joint accuracy:     96.6%
  AUC-ROC:            0.989
  F1:                 0.966
  Spatial-only:       96.6%
  Freq-only:          58.6%
  Delta (Δ):          +0.0%  (freq branch contribution)
Results saved → ./results/results.csv  (dd_convnext_base_scalar, acc=0.9664)



## Experiment 3: Gating Fusion

In [9]:
cfg3 = make_cfg("gating")
print(f"Running: {cfg3.experiment_name}")
train(cfg3, train_loader, val_loader, test_loader)


Running: dd_convnext_base_gating
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: dd_convnext_base_gating
Backbone: convnext_base | Fusion: gating | Frozen: False
Epochs: 20 | LR: 0.000279 | Batch: 88



Epoch   1/20 | train_loss=0.5406 | val_acc=99.1% | val_auc=1.000 | val_f1=0.990
  gate — entropy=1.750 nats | mean=0.482 | var=0.0069
  grad norms — freq=0.0184 | spatial=0.9997
  -> Saved best val_acc=99.1%


Epoch   2/20 | train_loss=0.4162 | val_acc=99.2% | val_auc=1.000 | val_f1=0.991
  gate — entropy=1.204 nats | mean=0.552 | var=0.0014
  -> Saved best val_acc=99.2%


Epoch   3/20 | train_loss=0.4079 | val_acc=99.2% | val_auc=1.000 | val_f1=0.992
  gate — entropy=1.561 nats | mean=0.473 | var=0.0062
  -> Saved best val_acc=99.2%


Epoch   4/20 | train_loss=0.3943 | val_acc=99.3% | val_auc=1.000 | val_f1=0.992
  gate — entropy=1.455 nats | mean=0.092 | var=0.0060
  -> Saved best val_acc=99.3%


...

Epoch  17/20 | train_loss=0.3640 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997
  gate — entropy=1.803 nats | mean=0.684 | var=0.0095


Epoch  18/20 | train_loss=0.3625 | val_acc=99.6% | val_auc=1.000 | val_f1=0.996
  gate — entropy=1.579 nats | mean=0.637 | var=0.0037


Epoch  19/20 | train_loss=0.3705 | val_acc=99.7% | val_auc=1.000 | val_f1=0.996
  gate — entropy=1.396 nats | mean=0.621 | var=0.0023


Epoch  20/20 | train_loss=0.3720 | val_acc=99.7% | val_auc=1.000 | val_f1=0.997
  gate — entropy=1.372 nats | mean=0.617 | var=0.0022

Training complete. Best val accuracy: 99.8%
Results will be logged to CSV after full_evaluation().


0.9980089963866972

In [10]:
results3 = full_evaluation(
    cfg3,
    checkpoint_path=f"./checkpoints/best_{cfg3.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_convnext_base_gating.pt

EVALUATION — dd_convnext_base_gating
Backbone: convnext_base | Fusion: gating | Frozen: False
  Joint accuracy:     96.7%
  AUC-ROC:            0.984
  F1:                 0.966
  Spatial-only:       96.7%
  Freq-only:          56.9%
  Delta (Δ):          +0.0%  (freq branch contribution)

  Gate distribution:
    entropy:  2.118 nats  (OK)
    mean:     0.510
    variance: 0.0165
Results saved → ./results/results.csv  (dd_convnext_base_gating, acc=0.967)



## Experiment 4: Gating Frozen

In [11]:
cfg4 = make_cfg("gating", frozen=True)
print(f"Running: {cfg4.experiment_name}")
train(cfg4, train_loader, val_loader, test_loader)


Running: dd_convnext_base_gating_frozen
Device: cuda
GPU: NVIDIA RTX PRO 4000 Blackwell

Experiment: dd_convnext_base_gating_frozen
Backbone: convnext_base | Fusion: gating | Frozen: True
Epochs: 20 | LR: 0.000279 | Batch: 88



Epoch   1/20 | train_loss=0.8248 | val_acc=90.3% | val_auc=0.968 | val_f1=0.890
  gate — entropy=2.202 nats | mean=0.443 | var=0.0122
  grad norms — freq=0.1264 | spatial=0.8986
  -> Saved best val_acc=90.3%


Epoch   2/20 | train_loss=0.7213 | val_acc=90.6% | val_auc=0.977 | val_f1=0.891
  gate — entropy=2.468 nats | mean=0.465 | var=0.0222
  -> Saved best val_acc=90.6%


Epoch   3/20 | train_loss=0.6847 | val_acc=91.7% | val_auc=0.980 | val_f1=0.906
  gate — entropy=2.669 nats | mean=0.486 | var=0.0329
  -> Saved best val_acc=91.7%


Epoch   4/20 | train_loss=0.6551 | val_acc=92.3% | val_auc=0.984 | val_f1=0.913
  gate — entropy=2.849 nats | mean=0.516 | var=0.0520
  -> Saved best val_acc=92.3%


...

Epoch  17/20 | train_loss=0.5146 | val_acc=93.6% | val_auc=0.990 | val_f1=0.927
  gate — entropy=2.889 nats | mean=0.540 | var=0.0629


Epoch  18/20 | train_loss=0.5099 | val_acc=93.5% | val_auc=0.990 | val_f1=0.926
  gate — entropy=2.891 nats | mean=0.536 | var=0.0650


Epoch  19/20 | train_loss=0.5075 | val_acc=93.3% | val_auc=0.990 | val_f1=0.923
  gate — entropy=2.884 nats | mean=0.542 | var=0.0627


Epoch  20/20 | train_loss=0.5074 | val_acc=93.3% | val_auc=0.990 | val_f1=0.922
  gate — entropy=2.885 nats | mean=0.542 | var=0.0632

Training complete. Best val accuracy: 94.4%
Results will be logged to CSV after full_evaluation().


0.9444731214512204

In [12]:
results4 = full_evaluation(
    cfg4,
    checkpoint_path=f"./checkpoints/best_{cfg4.experiment_name}.pt",
    test_loader=test_loader,
    dataset_type="deepdetect",
)


Loaded checkpoint: ./checkpoints/best_dd_convnext_base_gating_frozen.pt

EVALUATION — dd_convnext_base_gating_frozen
Backbone: convnext_base | Fusion: gating | Frozen: True
  Joint accuracy:     91.4%
  AUC-ROC:            0.971
  F1:                 0.910
  Spatial-only:       85.9%
  Freq-only:          62.3%
  Delta (Δ):          +5.4%  (freq branch contribution)

  Gate distribution:
    entropy:  2.897 nats  (OK)
    mean:     0.500
    variance: 0.0617

  No warning signs triggered. ✓
Results saved → ./results/results.csv  (dd_convnext_base_gating_frozen, acc=0.9136)



## Summary

In [20]:
import pandas as pd
df = pd.read_csv("./results/results.csv", on_bad_lines='skip', engine='python')
mask = df["backbone"] == BACKBONE
cols = ["experiment_name", "fusion", "frozen", "accuracy", "auc_roc", "f1", "gate_entropy"]
print(df[mask][cols].to_string(index=False))


               experiment_name     fusion  frozen  accuracy  auc_roc     f1  gate_entropy
            freq_only_baseline     gating   False    0.9436   0.9871 0.9446        0.0000
   dd_convnext_base_joint_only joint_only   False    0.9702   0.9872 0.9696        0.0000
       dd_convnext_base_scalar     scalar   False    0.9664   0.9886 0.9660        0.0000
       dd_convnext_base_gating     gating   False    0.9670   0.9842 0.9665        2.1183
dd_convnext_base_gating_frozen     gating    True    0.9136   0.9713 0.9095        2.8970
